# Hybrid RAG: a simple, step-by-step example

**RAG** (Retrieval-Augmented Generation) = first **search** your documents for the parts that match a question,
then give those parts to an LLM and ask it to answer **from them**.

**Hybrid RAG** = do that search in **two ways at once** and combine the results.

| Search | How it finds chunks | Good at | Weak at |
|---|---|---|---|
| **Keyword search** (BM25) | counts the question's **exact words** in each chunk | exact terms: years (*2019*), codes (*HR-LV-2022*), names | different wording: *"look after a sick parent"* won't match *"Caregiver leave"* |
| **Meaning search** (vectors) | compares the **meaning** of the question and each chunk | paraphrases and synonyms | exact numbers and codes: *2019* and *2020* look almost the same to it |

Each search is strong where the other is weak, so together they find the right chunk more often.

**Our documents:** `Leave_Policy` has **12 leave policies (2015–2026)** from one company. They are almost identical;
only the numbers change from year to year. A question like *"earned leave in 2019?"* matches **all 12** by meaning;
only keyword search can tell 2019 apart. That makes this a good dataset to see hybrid search work.

### Steps
| Step | What happens |
|---|---|
| 1 | Imports and settings |
| 2 | Read the PDFs |
| 3 | Split the text into chunks |
| 4 | Build the **keyword** index (BM25) |
| 5 | Build the **meaning** index (embeddings in Pinecone) |
| 6 | Keyword search function |
| 7 | Meaning search function |
| 8 | **Combine** both results (Reciprocal Rank Fusion) |
| 9 | Compare the three searches side by side |
| 10 | Connect to Gemini |
| 11 | Answer questions with Hybrid RAG |

### Step 1: Imports and settings

**What this cell does:** loads the libraries and sets a few values used later.

| Library | Used for |
|---|---|
| `pypdf` | reading text from the PDFs |
| `RecursiveCharacterTextSplitter` | cutting text into small chunks |
| `BM25Okapi` (rank_bm25) | keyword search |
| `SentenceTransformer` | turning text into vectors for meaning search |
| `Pinecone` | the database that stores the vectors and searches them |
| `genai` | Google Gemini, which writes the final answer |

**Settings**
- `DATA_FOLDER = "../Leave_Policy"`: where the PDFs are. `..` means *one folder up*, because this notebook is inside `Hybrid_RAG/`.
- `NAMESPACE = "hybrid-rag"`: our own section inside the Pinecone index, so we don't mix with other notebooks' data.
- `load_dotenv(find_dotenv(usecwd=True))`: loads the API keys from the `.env` file, searching from this folder upwards.

**Output:** nothing.

In [20]:
import os
import re
import json
import time

from pypdf import PdfReader
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from google import genai

load_dotenv(find_dotenv(usecwd=True))

DATA_FOLDER = "../Leave_Policy"
NAMESPACE = "hybrid-rag"

### Step 2: Read the PDFs

**What this cell does:** reads every leave-policy PDF, page by page.

1. Opens `manifest.json`, a list of the policies with each one's `year`, `file` name and document id (`doc_id`, e.g. `HR-LV-2019`).
2. For each policy, opens the PDF and takes the text of every page (`page.extract_text()`).
3. Saves each page with its **year** and **page number**, so later we can say where an answer came from.

**Output:** how many pages were read, and the start of the first page.

In [21]:
with open(os.path.join(DATA_FOLDER, "manifest.json"), encoding="utf-8") as f:
    manifest = json.load(f)

pages = []
for doc in manifest:
    reader = PdfReader(os.path.join(DATA_FOLDER, doc["file"]))
    for page_no, page in enumerate(reader.pages, start=1):
        pages.append({
            "year": doc["year"],
            "doc_id": doc["doc_id"],
            "page": page_no,
            "text": page.extract_text(),
        })

print(f"Read {len(pages)} pages from {len(manifest)} PDFs")
print("\nFirst page starts with:\n", pages[0]["text"][:300])

Read 47 pages from 12 PDFs

First page starts with:
 Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources 


### Step 3: Split the text into chunks

**What this cell does:** cuts every page into small pieces (**chunks**) of about 800 characters.

**Why chunks?** Searching works best on small, focused pieces. If a whole page were one piece, a question about one
sentence would compete with everything else on that page.

- `chunk_size=800`: each chunk is at most about 800 characters.
- `chunk_overlap=100`: neighbouring chunks share up to 100 characters, so a sentence cut at the border isn't lost.

**Important trick: a label in front of every chunk.**
```
Leave Policy 2019 (HR-LV-2019), page 2: Employees receive 10 days of Casual Leave ...
```
A chunk from the middle of a page might not mention its year at all. The label makes sure **both** searches always
know which year's policy a chunk belongs to.

Each chunk gets an id like `2019-p2-c1` (= 2019 policy, page 2, chunk 1).
`chunk_by_id` lets us look up a chunk's text from its id later.

**Output:** the number of chunks and one example.

In [22]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

chunks = []
for p in pages:
    for i, piece in enumerate(splitter.split_text(p["text"])):
        chunks.append({
            "id": f"{p['year']}-p{p['page']}-c{i}",
            "year": p["year"],
            "page": p["page"],
            "text": f"Leave Policy {p['year']} ({p['doc_id']}), page {p['page']}: {piece}",
        })

chunk_by_id = {c["id"]: c for c in chunks}

print(f"Created {len(chunks)} chunks\n")
print("Example chunk:", chunks[0]["id"])
print(chunks[0]["text"][:400])

Created 147 chunks

Example chunk: 2015-p1-c0
Leave Policy 2015 (HR-LV-2015), page 1: Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources Department
Applies to
All permanent employees of Acme Techno


### Step 4: Build the keyword index (BM25)

**What this cell does:** prepares the keyword search.

**1. `tokenize(text)`: split text into search words**
- `text.lower()`: lowercase, so *Sick* and *sick* match.
- `re.findall(r"[a-z0-9]+", ...)`: keep only letters and digits. `HR-LV-2019` becomes `hr`, `lv`, `2019`.
- Remove **stopwords** (*the, of, what, how, many ...*): they appear everywhere and don't help find the right chunk.

Example: *"How many days of Earned Leave in 2019?"* → `['days', 'earned', 'leave', '2019']`

**2. `BM25Okapi(...)`: the keyword index**

BM25 is the standard keyword-search formula used by search engines. For a question, it gives every chunk a score:
- more of the question's words in the chunk → **higher** score
- **rare** words count more than common ones: *2019* (only in the 2019 policy) counts a lot, *leave* (in every chunk) counts very little
- long chunks don't win just by being long

The same `tokenize` is used for chunks (here) and questions (Step 6), so both are split the same way.

**Output:** the words of an example question.

In [26]:
STOPWORDS = {"a", "an", "the", "of", "in", "on", "for", "to", "and", "or", "is", "are", "was", "were",
             "what", "which", "how", "many", "much", "does", "do", "did", "can", "i", "my", "me",
             "it", "this", "that", "by", "with", "at", "as", "from", "per", "get", "got"}



def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOPWORDS]
    

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

print(tokenize("How many a days of Earned Leave did employees get in 2019?"))
print(chunks[3])
print(bm25.get_scores(tokenize("How many a days of Earned Leave did employees get in 2019?")))

['days', 'earned', 'leave', 'employees', '2019']
{'id': '2015-p2-c0', 'year': 2015, 'page': 2, 'text': 'Leave Policy 2015 (HR-LV-2015), page 2: Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015\nPage 2\nEmployees receive 8 days of Casual Leave in 2015. It is credited on 1 January and is pro-rated by\ncompleted months for employees joining during the year. Unused Casual Leave lapses on 31 December\n2015 and cannot be carried forward or encashed. Casual Leave of more than 3 days at a stretch is not\npermitted and must be combined with Earned Leave. Half-day leave: Not permitted; minimum leave unit is\n1 day.\n5. Sick Leave (2015)\nEmployees receive 8 days of Sick Leave in 2015, credited on 1 January and pro-rated for new joiners. A\nmedical certificate from a registered medical practitioner is required when Sick Leave exceeds 3 days.\nSick Leave carry forward: Not offered. Sick Leave cannot be encashed.\n6. Earned Leave (2015)'}
[2.92405598 2.69041013 2.15895184 4.394575

### Step 5: Build the meaning index (embeddings in Pinecone)

Meaning search needs every chunk turned into a vector and stored in a vector database. This step is split into
six small cells, **5.1 to 5.6**, one job each.

**What is an embedding?** A list of numbers (here 384) that represents the **meaning** of a text.
Texts with similar meaning get similar numbers, e.g. *"time off to care for a parent"* ≈ *"Caregiver leave"*.

| Cell | Job |
|---|---|
| 5.1 | load the embedding model |
| 5.2 | connect to Pinecone |
| 5.3 | clear old vectors from earlier runs |
| 5.4 | turn every chunk into a vector |
| 5.5 | upload the vectors to Pinecone |
| 5.6 | wait until Pinecone has saved them |

#### 5.1 Load the embedding model

**What this cell does:** loads `all-MiniLM-L6-v2`, the model that turns text into vectors.
- Small and free, and runs **on your computer** (no API key needed).
- Downloaded the first time (about 90 MB), then loaded from the local cache.
- Every text becomes a vector of **384 numbers**.

**Output:** the vector of a short example sentence: its length (384) and its first 5 numbers.

In [27]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

example = model.encode("Caregiver leave")
print("Vector length:", len(example))
print("First 5 numbers:", example[:5])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1215.82it/s]


Vector length: 384
First 5 numbers: [-0.01955347 -0.01959092  0.0015571   0.02956124  0.04665591]


#### 5.2 Connect to Pinecone

**What this cell does:** opens the vector database.
1. Logs in with `PineCone_API_Key` from `.env`.
2. Reads the index name from `Pinecone_Index_Name` (an **index** is Pinecone's name for a table of vectors).
3. Creates the index **only if it doesn't exist yet**:
   - `dimension=384`: must match the model from 5.1
   - `metric="cosine"`: compares vectors by the angle between them (1 = same meaning)
4. `pc.Index(...)` gives us `index`, which we use to upload and search.

**Output:** the index name.

In [28]:
pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))
index_name = os.getenv("Pinecone_Index_Name")

if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=384, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index = pc.Index(index_name)
print("Connected to Pinecone index:", index_name)

Connected to Pinecone index: agenticai-training


#### 5.3 Clear old vectors

**What this cell does:** deletes all vectors in **our namespace** (`hybrid-rag`) left over from earlier runs, so we start clean.
- Only our namespace is cleared. Other notebooks' data in the same index is not touched.
- The very first time, the namespace doesn't exist yet and Pinecone raises an error; `try / except` simply ignores it.

**Output:** a confirmation line.

In [29]:
try:
    index.delete(delete_all=True, namespace=NAMESPACE)
except Exception:
    pass  # namespace doesn't exist yet (first run)

print(f"Namespace '{NAMESPACE}' is empty")

Namespace 'hybrid-rag' is empty


#### 5.4 Turn every chunk into a vector

**What this cell does:** sends the text of **all** chunks to the model in one call (`model.encode`).
The result is a table with one row per chunk and 384 numbers per row.

**Output:** the table's shape, e.g. `(147, 384)` = 147 chunks × 384 numbers.

In [30]:
vectors = model.encode([c["text"] for c in chunks])

print("Shape:", vectors.shape)

Shape: (147, 384)


#### 5.5 Upload the vectors to Pinecone

**What this cell does:** stores every vector in Pinecone.
1. Builds one **record** per chunk:
   - `id`: the chunk id (e.g. `2019-p2-c1`), so search results can be matched back to the chunk text
   - `values`: the 384 numbers (`.tolist()` turns them into a normal Python list)
   - `metadata`: extra information saved with the vector; here the **year**, so we could filter by year later
2. Uploads them in groups of 100 with `upsert` (= insert, or replace if the id already exists).
   Sending in groups keeps each request small.

**Output:** how many records were sent.

In [31]:
records = []
for i, c in enumerate(chunks):
    records.append({
        "id": c["id"],
        "values": vectors[i].tolist(),
        "metadata": {"year": c["year"]},
    })

for start in range(0, len(records), 100):
    index.upsert(vectors=records[start:start + 100], namespace=NAMESPACE)

print(f"Uploaded {len(records)} vectors")

Uploaded 147 vectors


#### 5.6 Wait until Pinecone has saved them

**What this cell does:** Pinecone needs a few seconds before new vectors can be searched. If we searched straight away,
some chunks could be missing from the results.

- `count_vectors()` asks Pinecone how many vectors are stored in our namespace.
- The loop checks every 2 seconds until the count equals the number of chunks (giving up after 60 seconds).

**Output:** how many vectors are stored. It should equal the number of chunks from Step 3.

In [32]:
def count_vectors():
    ns = index.describe_index_stats().namespaces.get(NAMESPACE)
    return ns.vector_count if ns else 0


for _ in range(30):
    if count_vectors() == len(chunks):
        break
    time.sleep(2)

print(f"Stored {count_vectors()} of {len(chunks)} vectors in Pinecone")

Stored 147 of 147 vectors in Pinecone


### Step 6: Keyword search

**What `keyword_search(question)` does:**
1. Splits the question into words with `tokenize` (Step 4).
2. `bm25.get_scores(...)` gives **every** chunk a score for those words.
3. Sorts the chunks from highest to lowest score and returns the top `k` (default 20) as `(chunk_id, score)`.

**Output:** the top 3 keyword results for a test question.

In [38]:
def keyword_search(question, k=20):
    scores = bm25.get_scores(tokenize(question))
    ranked = sorted(zip(chunks, scores), key=lambda pair: pair[1], reverse=True)
    return [(c["id"], score) for c, score in ranked[:k]]


for chunk_id, score in keyword_search("earned leave in 2019")[:3]:
    print(f"{score:6.2f}  {chunk_id}")

  6.73  2019-p2-c2
  6.70  2019-p4-c0
  6.55  2019-p1-c0


### Step 7: Meaning search

**What `vector_search(question)` does:**
1. Turns the question into a vector with the **same** model used for the chunks (Step 5).
2. Asks Pinecone for the `k` (default 20) most similar chunk vectors.
3. Returns them as `(chunk_id, score)`. The score is the cosine similarity (closer to 1 = more similar meaning).

**Output:** the top 3 meaning results for the same test question as Step 6. Compare the years.

In [39]:
def vector_search(question, k=20):
    result = index.query(vector=model.encode(question).tolist(), top_k=k, namespace=NAMESPACE)
    return [(match.id, match.score) for match in result.matches]


for chunk_id, score in vector_search("earned leave in 2019")[:3]:
    print(f"{score:6.3f}  {chunk_id}")

 0.681  2019-p2-c2
 0.643  2018-p2-c2
 0.637  2019-p1-c2


### Step 8: Combine both results (Reciprocal Rank Fusion)

This is the **heart of Hybrid RAG**: merge the keyword list and the meaning list into **one** ranking.

**Problem:** the scores can't be added directly. Keyword scores are roughly 0–30, meaning scores 0–1.

**Solution: Reciprocal Rank Fusion (RRF).** Ignore the scores and use only each chunk's **position** (rank) in each list:

```
RRF score = 1 / (60 + rank in keyword list)  +  1 / (60 + rank in meaning list)
```

| Chunk | Keyword rank | Meaning rank | RRF score |
|---|---|---|---|
| A | 1 | 2 | 1/61 + 1/62 = **0.0325** ← in both lists, so it wins |
| B | 2 | not found | 1/62 = 0.0161 |
| C | not found | 1 | 1/61 = 0.0164 |

A chunk that **both** searches like gets two contributions and rises to the top.
`60` is the standard value; it stops a single #1 rank from dominating.

**`hybrid_search(question)`** runs both searches, merges them with RRF, and returns the top 5 chunks.

**Output:** nothing (used in the next steps).

In [40]:
def hybrid_search(question, top_k=5):
    rrf_scores = {}
    for results in [keyword_search(question), vector_search(question)]:
        for rank, (chunk_id, _score) in enumerate(results, start=1):
            rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (60 + rank)

    ranked = sorted(rrf_scores.items(), key=lambda pair: pair[1], reverse=True)
    return ranked[:top_k]

### Step 9: Compare keyword vs meaning vs hybrid

**What this cell does:** for each test question, prints the **top 3** results of each search, showing the **year** and
**page** of each chunk, so you can see where each search succeeds or fails.

**What to look for:**

| Question | Keyword search | Meaning search | Hybrid |
|---|---|---|---|
| *earned leave in **2019**?* | all 2019 ✔ | mixes in another year (2018) ✘ | all 2019 ✔ |
| *time off to **look after a sick family member**?* | unrelated chunks: none of these words is in "Caregiver leave" ✘ | family-related leave ✔ | a **caregiver leave** chunk in the top 3 ✔ |
| *casual leave in policy **HR-LV-2022**?* | all 2022 ✔ | no 2022 chunk: the code means nothing to it ✘ | a **2022** chunk on top ✔ |

Each single search fails at least once; hybrid gets a useful result for all three. That's why we combine them.

(Scores can vary slightly between runs; the pattern stays the same.)

In [41]:
def show(chunk_id, score):
    c = chunk_by_id[chunk_id]
    text = c["text"].split(": ", 1)[1][:60].replace("\n", " ")
    print(f"     {score:7.3f}  year {c['year']}  page {c['page']}  | {text}")


def compare(question):
    print("QUESTION:", question)
    print("  Keyword search:")
    for chunk_id, score in keyword_search(question)[:3]:
        show(chunk_id, score)
    print("  Meaning search:")
    for chunk_id, score in vector_search(question)[:3]:
        show(chunk_id, score)
    print("  Hybrid search:")
    for chunk_id, score in hybrid_search(question)[:3]:
        show(chunk_id, score)
    print()


compare("How many days of earned leave did employees get in 2019?")
compare("Can I take time off to look after a sick family member?")
compare("What is the casual leave entitlement in policy HR-LV-2022?")

QUESTION: How many days of earned leave did employees get in 2019?
  Keyword search:
       8.963  year 2019  page 2  | a weekly off or public holiday; must be availed within 60 da
       7.901  year 2019  page 1  | Acme Technologies India Pvt. Ltd. | Leave Policy 2019 | HR-L
       7.762  year 2019  page 4  | Acme Technologies India Pvt. Ltd. | Leave Policy 2019 | HR-L
  Meaning search:
       0.660  year 2019  page 2  | Sick Leave carry forward: Not offered. Sick Leave cannot be 
       0.659  year 2019  page 1  | - Paternity leave: changed from 5 days to 7 days. - Adoption
       0.635  year 2018  page 1  | - Work from home policy: INTRODUCED - Ad-hoc, up to 2 days p
  Hybrid search:
       0.032  year 2019  page 2  | Sick Leave carry forward: Not offered. Sick Leave cannot be 
       0.031  year 2019  page 1  | - Paternity leave: changed from 5 days to 7 days. - Adoption
       0.030  year 2019  page 3  | 11. Applying for and Approving Leave (2019) How to apply: On

QUESTION: Can I

### Step 10: Connect to Google Gemini

**What this cell does:** connects to Gemini with `GEMINI_API_KEY` from `.env`, and defines `ask_gemini(prompt)`,
which sends a prompt and returns the reply text.

If you see **503 UNAVAILABLE**, Google's servers are busy: wait a minute and run the cell again,
or change the model name (e.g. to `gemini-3.5-flash-lite`).

**Output:** nothing.

In [42]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


def ask_gemini(prompt):
    response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return response.text

### Step 11: Answer questions with Hybrid RAG

**What `answer(question)` does:** the full Hybrid RAG pipeline in 4 steps.

1. **Search:** `hybrid_search(question)` gets the 5 best chunks (Step 8).
2. **Build the context:** the chunks are joined together, each labelled `[Source 1]`, `[Source 2]` ...
3. **Build the prompt:** the instructions + the context + the question. The instructions tell Gemini to:
   - answer **only** from the sources (not from its own knowledge)
   - use the **year** the question asks about, since the numbers change every year
   - cite the sources it used, e.g. `[Source 2]`
   - say so if the answer isn't in the sources
4. **Ask Gemini** and print the answer, plus which year and page each source came from.

**Output:** the answers to the questions below.

In [43]:
def answer(question):
    results = hybrid_search(question)

    context = ""
    for n, (chunk_id, _score) in enumerate(results, start=1):
        context += f"[Source {n}] {chunk_by_id[chunk_id]['text']}\n\n"

    prompt = f"""Answer the question using only the sources below.
These are leave policies for different years; use the policy of the year the question asks about.
If no year is asked, use the latest year in the sources and say which year.
Cite the sources you used, e.g. [Source 2]. If the answer is not in the sources, say so.

{context}
Question: {question}"""

    print("Question:", question)
    print("\nAnswer:", ask_gemini(prompt))
    print("\nSources:")
    for n, (chunk_id, _score) in enumerate(results, start=1):
        c = chunk_by_id[chunk_id]
        print(f"  [Source {n}] Leave Policy {c['year']}, page {c['page']}")
    print("-" * 80)

**Question 1: an exact year.** Keyword search finds *2019*. Expected: **18 days**.

In [44]:
answer("How many days of earned leave did employees get in 2019?")

Question: How many days of earned leave did employees get in 2019?

Answer: In 2019, employees received **18 days** of Earned Leave (also called Privilege Leave) [Source 1, Source 2].

Sources:
  [Source 1] Leave Policy 2019, page 2
  [Source 2] Leave Policy 2019, page 1
  [Source 3] Leave Policy 2019, page 3
  [Source 4] Leave Policy 2019, page 3
  [Source 5] Leave Policy 2019, page 2
--------------------------------------------------------------------------------


**Question 2: different wording.** The policy calls it *Caregiver leave*; meaning search finds it. Expected: **caregiver leave, 5 days per year**.

In [ ]:
answer("Can I take time off to look after a sick family member?")

**Question 3: an exact leave name.** Keyword search finds *fertility treatment*; it appears only in recent policies. Expected: **introduced in 2026, 5 days per year**.

In [ ]:
answer("When was fertility treatment leave introduced and how many days does it give?")